[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C73_3D_Representation_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与「三个乘子」

这个 notebook 做三件事：

1. **造一个合成路口点云**（地面 + 4 辆车 + 3 块标志 + 散点），全课复用。
2. **把四种表示的代价算成具体数字**：
   $r{=}0.05$ m 的密集体素是 **640 MB**，而同一场景的点云是 **1.44 MB**——
   **而那 640 MB 里只有 0.018% 非空，平均每格 1.03 个点。**
3. **量出「三个乘子」**：网格规模 $(L/r)^3$、卷积核 $k^3$、IoU 的三个因子。
   同样每轴 10% 的相对误差，**2D IoU 是 0.681 而 3D 只有 0.574**。

> 心智模型：**从 2D 到 3D，几乎每个量都从两个因子的积变成三个因子的积。
> 而这四件事互相加强——本课六个模块就是沿着这条链走。**

## 0 · 环境

In [ ]:
import numpy as np

print('numpy', np.__version__)
print('本课全程 CPU / 断网 / **不训练任何真实网络**')
print('前提：C72（多视角几何）—— 投影、标定、位姿本课直接使用，不再重讲')

# 场景范围（米）：X 前 100、Y 左右各 50、Z 上 8  —— 沿用 C72 的车体坐标约定
SCENE_L = (100.0, 100.0, 8.0)
N_PTS = 120_000                     # 一帧 64 线 LiDAR 的量级

## 1 · 合成路口点云

**刻意简化但几何上合理**：没有真实 LiDAR 的射线模型（无遮挡阴影、
点密度不随距离衰减、无反射率）。所以本课关于占用率的**相对**结论可迁移，
具体百分比只属于这个配置。

In [ ]:
def synth_scene(n=N_PTS, seed=0):
    """合成路口：70% 地面 + 4 辆车 + 3 块标志 + 其余散点。"""
    r = np.random.default_rng(seed)
    parts = []

    n_g = int(n * 0.70)
    parts.append(np.column_stack([r.uniform(0, 100, n_g),
                                  r.uniform(-50, 50, n_g),
                                  r.normal(0, 0.02, n_g)]))          # 地面

    for cx, cy in [(20, -3), (35, 3), (60, -3), (80, 3)]:             # 车
        m = int(n * 0.05)
        parts.append(np.column_stack([r.uniform(cx - 2.2, cx + 2.2, m),
                                      r.uniform(cy - 0.9, cy + 0.9, m),
                                      r.uniform(0, 1.5, m)]))

    for cx, cy in [(30, -5.5), (55, 5.5), (75, -5.5)]:                # 标志（又小又薄）
        m = int(n * 0.01)
        parts.append(np.column_stack([r.uniform(cx - 0.05, cx + 0.05, m),
                                      r.uniform(cy - 0.4, cy + 0.4, m),
                                      r.uniform(1.8, 2.6, m)]))

    used = sum(len(p) for p in parts)
    k = n - used
    parts.append(np.column_stack([r.uniform(0, 100, k),
                                  r.uniform(-50, 50, k),
                                  r.uniform(0, 8, k)]))               # 散点
    return np.vstack(parts)

P = synth_scene()
print(f'点数 {len(P):,}')
print(f'范围 X [{P[:,0].min():6.2f}, {P[:,0].max():6.2f}]  '
      f'Y [{P[:,1].min():6.2f}, {P[:,1].max():6.2f}]  '
      f'Z [{P[:,2].min():5.2f}, {P[:,2].max():5.2f}]')
assert len(P) == N_PTS
# 地面点占大头，这与真实 LiDAR 一致
near_ground = (np.abs(P[:, 2]) < 0.1).mean()
print(f'|z| < 0.1 m 的点占 {near_ground:.1%}  ← 地面主导，真实点云也是这样')
assert near_ground > 0.6

## 2 · 四种表示的内存账

$(L/r)^3$ 这个乘子的直接后果。

In [ ]:
def dense_voxel_count(L, r):
    return int(np.prod([l / r for l in L]))

pc_bytes = len(P) * 3 * 4          # float32
print(f'点云:  {len(P):>10,} 点 × 3 × float32 = {pc_bytes/1e6:8.2f} MB\n')
print(f"{'r (m)':>7s} {'格数':>15s} {'占据字节(1B/格)':>16s} {'相对点云':>10s}")
mem = {}
for r in [0.5, 0.2, 0.1, 0.05]:
    n = dense_voxel_count(SCENE_L, r)
    mem[r] = n
    print(f'{r:7.2f} {n:15,} {n/1e6:15.2f} MB {n/pc_bytes:9.1f}×')

assert mem[0.05] / mem[0.1] == 8, '格数应当按 r³ 增长：r 减半 -> 8 倍'
assert abs(mem[0.05] / pc_bytes - 444) < 5
print(f'\n✅ r 减半 -> 格数 ×8（这就是第一个乘子）')
print(f'   r=0.05 m 的密集体素是点云的 {mem[0.05]/pc_bytes:.0f} 倍内存')

# 网格与隐式表示的量级（对照）
print(f'\n对照：')
print(f'  网格 mesh:  假设 5 万顶点 + 10 万面 = '
      f'{(50_000*3*4 + 100_000*3*4)/1e6:.2f} MB')
print(f'  隐式 SDF:   一个 8×256 的 MLP ≈ '
      f'{(3*256 + 7*256*256 + 256)*4/1e6:.2f} MB（**与分辨率无关**）')

## 3 · 占用率：那 640 MB 里有多少是空的

**注意最后一列**——精细体素化之后「平均每格点数」掉到 1.03，
也就是**体素化已经不再压缩任何东西**。

In [ ]:
def voxel_stats(pts, r, L=SCENE_L):
    idx = np.floor(pts / r).astype(np.int64)
    uniq = np.unique(idx, axis=0)
    total = dense_voxel_count(L, r)
    return {'occupied': len(uniq), 'total': total,
            'rate': len(uniq) / total, 'pts_per_voxel': len(pts) / len(uniq)}

print(f"{'r (m)':>7s} {'非空格':>10s} {'总格数':>15s} {'占用率':>10s} {'平均每格点数':>13s}")
st = {}
for r in [0.5, 0.2, 0.1, 0.05]:
    s = voxel_stats(P, r)
    st[r] = s
    print(f'{r:7.2f} {s["occupied"]:10,} {s["total"]:15,} '
          f'{100*s["rate"]:9.4f}% {s["pts_per_voxel"]:12.2f}')

assert st[0.5]['rate'] > 0.05 and st[0.05]['rate'] < 0.0005
print(f'\n占用率从 {100*st[0.5]["rate"]:.2f}% 掉到 {100*st[0.05]["rate"]:.4f}%'
      f' —— 降了 {st[0.5]["rate"]/st[0.05]["rate"]:.0f} 倍')

# 关键：平均每格点数趋于 1
assert st[0.05]['pts_per_voxel'] < 1.1, '精细体素化时每格几乎只有一个点'
print(f'而平均每格点数从 {st[0.5]["pts_per_voxel"]:.2f} 掉到 '
      f'{st[0.05]["pts_per_voxel"]:.2f}')
print('✅ **精细体素化几乎等于点云** —— 它不再压缩，只是套上一层空格子')
print('   → 「体素分辨率越高越好」在某个点之后是纯浪费，'
      '而那个点由「平均每格点数」找出来')

## 4 · 量化误差的两个精确常数

点被搬到格中心，位移有一个精确上界和一个精确的均匀分布均值。

In [ ]:
def quant_error(pts, r):
    idx = np.floor(pts / r)
    centers = (idx + 0.5) * r
    d = np.linalg.norm(pts - centers, axis=1)
    return d

# 常数一：最大位移 = r√3/2（立方体中心到顶点）
# 常数二：均匀分布在立方体内时的平均距离 = 0.4804 r（数值积分得到）
u = np.random.default_rng(1).uniform(-0.5, 0.5, (400_000, 3))
C_UNIFORM = float(np.linalg.norm(u, axis=1).mean())
print(f'均匀立方体（边长 1）内到中心的平均距离 = {C_UNIFORM:.4f}')
print(f'  最大 = {np.linalg.norm(u,axis=1).max():.4f}   理论 √3/2 = {np.sqrt(3)/2:.4f}')
assert abs(C_UNIFORM - 0.4804) < 0.002, C_UNIFORM

print(f"\n{'r (m)':>7s} {'理论上界 r√3/2':>15s} {'实测最大':>10s} "
      f"{'实测平均':>10s} {'均匀理论 0.4804r':>17s} {'实测/均匀':>10s}")
for r in [0.5, 0.2, 0.1, 0.05]:
    d = quant_error(P, r)
    ub = r * np.sqrt(3) / 2
    print(f'{r:7.2f} {ub:14.4f}m {d.max():9.4f}m {d.mean():9.4f}m '
          f'{C_UNIFORM*r:16.4f}m {d.mean()/(C_UNIFORM*r):9.2f}')
    assert d.max() <= ub + 1e-12, '不可能超过上界'

# 实测平均比均匀理论**大**，因为地面点在 z 上集中在格子边界附近
r = 0.5
d = quant_error(P, r)
ratio = d.mean() / (C_UNIFORM * r)
print(f'\n实测平均是均匀理论的 {ratio:.2f} 倍')
assert ratio > 1.1, '点在体素内不是均匀分布的'
print('✅ 上界 r√3/2 是硬的；而**实测均值大于均匀理论值**——')
print('   因为 70% 的点是地面点，它们在 z 上集中在格子的某个面附近，')
print('   而「贴在一个面上」的平均距离是 0.6246r（> 0.4804r）')
u2 = np.column_stack([np.random.default_rng(2).uniform(-0.5,0.5,200_000),
                      np.random.default_rng(3).uniform(-0.5,0.5,200_000),
                      np.full(200_000, -0.48)])
print(f'   验证：贴面点的平均距离 = {np.linalg.norm(u2,axis=1).mean():.4f} '
      f'（均匀是 {C_UNIFORM:.4f}）')

## 5 · 三个乘子之三：IoU

$\text{IoU}$ 在 3D 里是三个因子的积，所以**同样的相对误差在 3D 下更痛**。

In [ ]:
def iou_axis_aligned(size, delta):
    """同尺寸轴对齐框，中心相距 delta 时的 IoU（维数由 size 决定）。"""
    s = np.asarray(size, float); d = np.abs(np.asarray(delta, float))
    inter = np.prod(np.maximum(s - d, 0.0))
    vol = np.prod(s)
    den = 2 * vol - inter
    return inter / den if den > 0 else 0.0

print('单位立方体 / 单位正方形，每轴相对误差都是 rel：\n')
print(f"{'rel':>6s} {'2D IoU':>9s} {'3D IoU':>9s} {'3D/2D':>8s}")
for rel in [0.02, 0.05, 0.10, 0.20]:
    i2 = iou_axis_aligned((1., 1.), (rel, rel))
    i3 = iou_axis_aligned((1., 1., 1.), (rel, rel, rel))
    print(f'{rel:6.0%} {i2:9.3f} {i3:9.3f} {i3/i2:8.3f}')
    assert i3 < i2, '同样的相对误差，3D IoU 必然更低'

i2 = iou_axis_aligned((1., 1.), (0.1, 0.1))
i3 = iou_axis_aligned((1., 1., 1.), (0.1, 0.1, 0.1))
print(f'\n每轴 10%: 2D {i2:.3f} vs 3D {i3:.3f}')
assert abs(i2 - 0.681) < 0.002 and abs(i3 - 0.574) < 0.002
print('✅ 三个乘子的直接后果：**2D 的 IoU 阈值不能照搬到 3D**')

# 卷积核也是同一个乘子
print(f'\n卷积核：3×3 = {3**2} vs 3×3×3 = {3**3}  → {3**3/3**2:.0f}×')
Cin = Cout = 64
dense_flops = st[0.1]['total'] * 27 * Cin * Cout
print(f'r=0.1 m 的密集 3³ 卷积（{Cin}→{Cout}）= {dense_flops/1e9:.0f} GFLOP')
print(f'  而其中 {100*(1-st[0.1]["rate"]):.2f}% 的算力花在空格子上（模块 03 会拿回来）')

## 6 · 小结

| 乘子 | 数值 | 在哪个模块被展开 |
|---|---|---|
| 网格规模 $(L/r)^3$ | $r{=}0.05$ m → 6.4 亿格 / **640 MB**（点云的 444×） | 01 |
| 占用率 | 9.43% → **0.018%**（降 518 倍） | 01 / 03 |
| **平均每格点数** | 1.99 → **1.03**（精细体素化不再压缩） | 01 |
| 量化误差上界 | $r\sqrt3/2$（硬上界，实测吻合） | 01 |
| 均匀分布均值 | **$0.4804r$**；实测在 $r{=}0.5$ m 时大 1.20 倍，随 $r$ 减小降到 1.00 | 01 |
| 卷积核 $k^3$ | 27 而不是 9；密集 3³ 卷积 **283 GFLOP** | 03 |
| **IoU 的三个乘子** | 每轴 10%：2D **0.681** vs 3D **0.574** | 05 |

## ✏️ 练习 1：体素化与统计

实现 `voxelize(pts, r, L=SCENE_L)`，返回 dict：

- `'idx'` —— `(M,3)` 的去重体素索引（按字典序排序）
- `'counts'` —— `(M,)` 每格的点数
- `'rate'` —— 占用率
- `'pts_per_voxel'` —— 平均每格点数
- `'saturated'` —— bool：`pts_per_voxel < 1.1`（**体素化已不再压缩**）

要求不用 Python 循环遍历点。

In [ ]:
def voxelize(pts, r, L=SCENE_L):
    """返回 dict(idx, counts, rate, pts_per_voxel, saturated)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
for r in [0.5, 0.2, 0.1, 0.05]:
    v = voxelize(P, r)
    assert set(v) == {'idx', 'counts', 'rate', 'pts_per_voxel', 'saturated'}
    assert v['idx'].shape[1] == 3
    assert len(v['idx']) == len(v['counts'])
    assert v['counts'].sum() == len(P), '每个点必须恰好落进一格'
    # 与第 3 节的独立实现一致
    ref = voxel_stats(P, r)
    assert len(v['idx']) == ref['occupied'], (r, len(v['idx']), ref['occupied'])
    assert abs(v['rate'] - ref['rate']) < 1e-15
    # 索引已排序且去重
    assert len(np.unique(v['idx'], axis=0)) == len(v['idx'])

print(f"{'r':>6s} {'非空格':>9s} {'占用率':>10s} {'每格点数':>10s} {'最多点数':>9s} {'饱和?':>7s}")
for r in [0.5, 0.2, 0.1, 0.05]:
    v = voxelize(P, r)
    print(f'{r:6.2f} {len(v["idx"]):9,} {100*v["rate"]:9.4f}% '
          f'{v["pts_per_voxel"]:9.2f} {v["counts"].max():9d} {str(v["saturated"]):>7s}')

assert voxelize(P, 0.5)['saturated'] is False
assert voxelize(P, 0.05)['saturated'] is True
print('\n✅ 练习 1 通过：`saturated` 一眼看出「再细分就是纯浪费」的分辨率')

## 📖 参考答案 1

In [ ]:
# 练习 1 参考答案
def voxelize(pts, r, L=SCENE_L):
    idx = np.floor(np.asarray(pts, float) / r).astype(np.int64)
    uniq, counts = np.unique(idx, axis=0, return_counts=True)
    total = dense_voxel_count(L, r)
    ppv = len(pts) / len(uniq)
    return {'idx': uniq, 'counts': counts,
            'rate': len(uniq) / total,
            'pts_per_voxel': ppv,
            'saturated': bool(ppv < 1.1)}

v = voxelize(P, 0.05)
assert v['counts'].sum() == len(P) and v['saturated'] is True
print('✅ 参考答案 1 通过（np.unique(axis=0, return_counts=True) 一次拿到去重与计数）')

## ✏️ 练习 2：量化误差的两个常数

实现 `quant_stats(pts, r)`，返回 dict：

- `'max'`, `'mean'` —— 实测的最大/平均位移
- `'upper_bound'` —— 理论上界 $r\sqrt3/2$
- `'uniform_mean'` —— 均匀分布的理论均值 $0.4804r$
- `'bound_ok'` —— bool：实测最大是否 $\le$ 上界
- `'nonuniformity'` —— 实测均值 / 均匀理论均值（**> 1 说明点在格内不均匀**）

> 自测会顺带发现一件事：**不均匀度随 $r$ 减小单调下降到 1**（1.22 → 1.00）。
> 它其实是一个免费的探针，度量「体素尺度**以下**还有多少结构」。

In [ ]:
def quant_stats(pts, r, c_uniform=0.4804):
    """返回 dict(max, mean, upper_bound, uniform_mean, bound_ok, nonuniformity)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
# 参照物：在一个格子里放均匀点，nonuniformity 应当接近 1
rg = np.random.default_rng(7)
UNI = rg.uniform(0, 0.5, (200_000, 3))          # 恰好落在 r=0.5 的一格里
s_uni = quant_stats(UNI, 0.5)
assert set(s_uni) == {'max', 'mean', 'upper_bound', 'uniform_mean',
                      'bound_ok', 'nonuniformity'}
assert s_uni['bound_ok'] is True
assert abs(s_uni['nonuniformity'] - 1.0) < 0.02, \
    f"均匀点的 nonuniformity 应当接近 1，实测 {s_uni['nonuniformity']:.3f}"
print(f"均匀参照: mean {s_uni['mean']:.4f}  理论 {s_uni['uniform_mean']:.4f}  "
      f"nonuniformity {s_uni['nonuniformity']:.3f}")

RS = [1.0, 0.5, 0.2, 0.1, 0.05, 0.02]
print(f"\n{'r':>6s} {'上界':>9s} {'实测max':>9s} {'实测mean':>10s} "
      f"{'均匀理论':>10s} {'不均匀度':>9s}")
nu = []
for r in RS:
    st2 = quant_stats(P, r)
    nu.append(st2['nonuniformity'])
    assert st2['bound_ok'] is True, f'r={r} 超过上界'
    print(f'{r:6.2f} {st2["upper_bound"]:8.4f}m {st2["max"]:8.4f}m '
          f'{st2["mean"]:9.4f}m {st2["uniform_mean"]:9.4f}m '
          f'{st2["nonuniformity"]:8.3f}')

# ① 不均匀度恒 >= 1（点不可能比均匀分布更靠近中心太多），且**随 r 单调下降到 1**
assert all(v > 0.99 for v in nu)
assert nu == sorted(nu, reverse=True), f'不均匀度应随 r 减小而单调下降：{nu}'
assert nu[0] > 1.2 and nu[-1] < 1.01, (nu[0], nu[-1])
print(f'\n不均匀度 {nu[0]:.3f} (r=1.0m) → {nu[-1]:.3f} (r=0.02m)，单调下降到 1')
print('  → 它度量的是「体素尺度**以下**还有多少结构」：')
print('    粗体素时地面点挤在某个面附近（1.22）；')
print(f'    而 r 小于地面抖动 σ=0.02m 的量级后，格内看起来就是均匀的（{nu[-1]:.3f}）')

# ② 上界随 r 严格线性
s1, s2 = quant_stats(P, 0.5), quant_stats(P, 0.25)
assert abs(s1['upper_bound'] / s2['upper_bound'] - 2.0) < 1e-12
print('\n✅ 练习 2 通过：上界 r√3/2 是硬的、线性的；'
      '而 nonuniformity 是一个免费的「亚体素结构」探针')

## 📖 参考答案 2

In [ ]:
# 练习 2 参考答案
def quant_stats(pts, r, c_uniform=0.4804):
    pts = np.asarray(pts, float)
    centers = (np.floor(pts / r) + 0.5) * r
    d = np.linalg.norm(pts - centers, axis=1)
    ub = r * np.sqrt(3) / 2
    um = c_uniform * r
    return {'max': float(d.max()), 'mean': float(d.mean()),
            'upper_bound': float(ub), 'uniform_mean': float(um),
            'bound_ok': bool(d.max() <= ub + 1e-12),
            'nonuniformity': float(d.mean() / um)}

assert quant_stats(P, 0.1)['bound_ok']
assert quant_stats(UNI, 0.5)['nonuniformity'] < 1.02
print('✅ 参考答案 2 通过')
print('   0.4804 不是随手写的常数：它是均匀立方体内到中心的平均距离，')
print('   本 notebook 第 4 节用 40 万个采样点验证过。')

## ✏️ 练习 3：表示选型器

实现 `pick_representation(need)`，`need` 是一个 dict：

| 键 | 含义 |
|---|---|
| `accuracy_m` | 需要的位置精度（米） |
| `needs_conv` | bool，是否要用卷积 |
| `memory_budget_mb` | 内存预算 |
| `scene_l` | 场景尺寸三元组 |
| `n_points` | 点数 |

返回 `(表示, 理由, 可用的最细体素分辨率或 None)`。规则：

1. `needs_conv` 为真 → 体素；此时算出**内存预算允许的最细 $r$**，
   并检查 $r\sqrt3/2 \le$ `accuracy_m`；不满足则返回 `'voxel_infeasible'`
2. 否则若点云内存 $\le$ 预算 → 点云
3. 否则 → 隐式

In [ ]:
def pick_representation(need):
    """返回 (representation, reason, finest_r or None)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
BASE = {'accuracy_m': 0.10, 'needs_conv': True, 'memory_budget_mb': 100.0,
        'scene_l': SCENE_L, 'n_points': N_PTS}

rep, why, r_fin = pick_representation(BASE)
print(f'需要卷积 + 100MB 预算 + 0.10m 精度 -> {rep} ({why}), 最细 r={r_fin}')
assert rep in ('voxel', 'voxel_infeasible')

print(f"\n{'精度要求':>9s} {'预算(MB)':>9s} {'要卷积':>7s} {'选择':>18s} {'最细 r':>9s}")
for acc in [0.30, 0.10, 0.02]:
    for bud in [10.0, 100.0, 2000.0]:
        for conv in [True, False]:
            n = dict(BASE, accuracy_m=acc, memory_budget_mb=bud, needs_conv=conv)
            rep, why, rf = pick_representation(n)
            rs = 'None' if rf is None else f'{rf:.4f}'
            print(f'{acc:8.2f}m {bud:9.0f} {str(conv):>7s} {rep:>18s} {rs:>9s}')

# ① 要卷积、预算够、精度松 -> 体素可行
r1 = pick_representation(dict(BASE, accuracy_m=0.30, memory_budget_mb=100.))
assert r1[0] == 'voxel' and r1[2] is not None
assert r1[2] * np.sqrt(3) / 2 <= 0.30 + 1e-12

# ② 要卷积、精度苛刻 -> 体素不可行（内存撑不住那么细的 r）
r2 = pick_representation(dict(BASE, accuracy_m=0.02, memory_budget_mb=100.))
assert r2[0] == 'voxel_infeasible', r2
print(f'\n0.02m 精度 + 100MB: {r2[0]} —— 需要 r <= '
      f'{0.02*2/np.sqrt(3):.4f}m，而预算只够 {r2[2]:.4f}m')

# ③ 不要卷积、点云装得下 -> 点云
r3 = pick_representation(dict(BASE, needs_conv=False, memory_budget_mb=100.))
assert r3[0] == 'point_cloud'

# ④ 不要卷积、连点云都装不下 -> 隐式
r4 = pick_representation(dict(BASE, needs_conv=False, memory_budget_mb=0.5))
assert r4[0] == 'implicit', r4
print('✅ 练习 3 通过：选型的关键是**把「精度要求」翻译成 r，再看内存装不装得下**')

## 📖 参考答案 3

In [ ]:
# 练习 3 参考答案
def pick_representation(need):
    L = need['scene_l']
    budget_bytes = need['memory_budget_mb'] * 1e6
    # 预算允许的最细 r：(L0/r)(L1/r)(L2/r) * 1 byte <= budget
    finest_r = (np.prod(L) / budget_bytes) ** (1.0 / 3.0)

    if need['needs_conv']:
        if finest_r * np.sqrt(3) / 2 <= need['accuracy_m'] + 1e-12:
            return 'voxel', 'conv_needed_and_affordable', float(finest_r)
        return ('voxel_infeasible',
                'accuracy_requires_finer_r_than_memory_allows', float(finest_r))

    pc_bytes = need['n_points'] * 3 * 4
    if pc_bytes <= budget_bytes:
        return 'point_cloud', 'exact_and_fits', None
    return 'implicit', 'resolution_independent_memory', None

assert pick_representation(dict(BASE, accuracy_m=0.30))[0] == 'voxel'
assert pick_representation(dict(BASE, accuracy_m=0.02))[0] == 'voxel_infeasible'
print('✅ 参考答案 3 通过')
print('   注意 finest_r 的解法：内存约束是 ∏(L_i/r) ≤ B，所以 r ≥ (∏L_i / B)^(1/3)')
print('   —— 而 1/3 次方正是「第一个乘子」的反面：预算翻 8 倍才能把 r 减半。')

## ✏️ 练习 4：三个乘子的换算器

实现 `multiplier_report(dims, rel_err)`，`dims` 是 2 元或 3 元的尺寸元组，
返回 dict：

- `'iou'` —— 每轴相对误差为 `rel_err` 时的 IoU
- `'grid_cells'` —— `{r: 格数}`，对 `r in (0.5, 0.1)`，场景取 `SCENE_L`
- `'kernel_ops'` —— $3^{\text{dim}}$
- `'max_rel_err_at_iou50'` —— 反解：IoU 恰好 0.5 时每轴允许的相对误差

然后用它对比 2D 与 3D。

In [ ]:
def multiplier_report(dims, rel_err, L=SCENE_L):
    """返回 dict(iou, grid_cells, kernel_ops, max_rel_err_at_iou50)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
r2 = multiplier_report((1., 1.), 0.10)
r3 = multiplier_report((1., 1., 1.), 0.10)
for r in (r2, r3):
    assert set(r) == {'iou', 'grid_cells', 'kernel_ops', 'max_rel_err_at_iou50'}

print(f"{'维度':>5s} {'IoU@10%':>9s} {'核算子':>7s} {'IoU=0.5 允许的每轴相对误差':>26s}")
for d, rr in [(2, r2), (3, r3)]:
    print(f'{d:4d}D {rr["iou"]:9.3f} {rr["kernel_ops"]:7d} '
          f'{rr["max_rel_err_at_iou50"]:25.1%}')

assert r3['iou'] < r2['iou'], '同样相对误差下 3D IoU 更低'
assert abs(r2['iou'] - 0.681) < 0.002 and abs(r3['iou'] - 0.574) < 0.002
assert r3['kernel_ops'] == 27 and r2['kernel_ops'] == 9
assert r3['max_rel_err_at_iou50'] < r2['max_rel_err_at_iou50'], \
    '3D 允许的误差更小'

# 反解出来的阈值代回去，应当正好给 0.5
for rr, dims in [(r2, (1., 1.)), (r3, (1., 1., 1.))]:
    e = rr['max_rel_err_at_iou50']
    got = iou_axis_aligned(dims, tuple([e] * len(dims)))
    assert abs(got - 0.5) < 1e-6, (dims, e, got)
print('\n反解自洽：把允许误差代回去都给出 IoU = 0.500')

# 网格规模：r 从 0.5 到 0.1 -> 2D 25 倍、3D 125 倍
g2 = r2['grid_cells']; g3 = r3['grid_cells']
print(f'\nr 0.5 -> 0.1 的格数增长: 2D {g2[0.1]/g2[0.5]:.0f}×  '
      f'3D {g3[0.1]/g3[0.5]:.0f}×')
assert abs(g2[0.1]/g2[0.5] - 25) < 1e-6
assert abs(g3[0.1]/g3[0.5] - 125) < 1e-6
print('✅ 练习 4 通过：三个乘子在一个函数里被同时算出来——')
print('   而它们解释了本课后面五个模块的全部设计动机')

## 📖 参考答案 4

In [ ]:
# 练习 4 参考答案
def multiplier_report(dims, rel_err, L=SCENE_L):
    n = len(dims)
    cells = {}
    for r in (0.5, 0.1):
        cells[r] = int(np.prod([L[i] / r for i in range(n)]))
    # 反解：同尺寸框、每轴相对误差 e -> IoU = (1-e)^n / (2 - (1-e)^n)
    # 令它 = 0.5  =>  (1-e)^n = 2/3  =>  e = 1 - (2/3)^(1/n)
    e50 = 1.0 - (2.0 / 3.0) ** (1.0 / n)
    return {'iou': iou_axis_aligned(dims, tuple([rel_err] * n)),
            'grid_cells': cells,
            'kernel_ops': 3 ** n,
            'max_rel_err_at_iou50': float(e50)}

for dims in [(1., 1.), (1., 1., 1.)]:
    rr = multiplier_report(dims, 0.10)
    assert abs(iou_axis_aligned(dims,
               tuple([rr['max_rel_err_at_iou50']] * len(dims))) - 0.5) < 1e-9
print('✅ 参考答案 4 通过')
print('   闭式解值得记：IoU = q/(2−q) 其中 q = (1−e)^n，')
print('   所以 IoU=0.5 ⇔ q=2/3 ⇔ e = 1 − (2/3)^(1/n)：')
print(f'     2D: {1-(2/3)**0.5:.1%}   3D: {1-(2/3)**(1/3):.1%}'
      f'   → **维数越高，允许的相对误差越小**')

## 🧪 真实工程胶囊

```python
# ── 1) 真实点云的读取（本课用合成场景，因为要知道真值）──
#   KITTI: 每点 4 个 float32 (x, y, z, intensity)
pts = np.fromfile(bin_path, dtype=np.float32).reshape(-1, 4)
#   nuScenes: 5 个 (x, y, z, intensity, ring_index)
#   ⚠️ 坐标系各家不同 —— 而 C72 模块 00 的第一条纪律是「坐标系约定先钉死」

# ── 2) 体素化：真实实现用哈希而不是 np.unique ──
#   spconv / MinkowskiEngine 内部都是哈希表 + 坐标键
import spconv.pytorch as spconv
voxels, coords, num_pts = spconv.utils.PointToVoxel(
    vsize_xyz=[0.05, 0.05, 0.1],          # ← 注意 z 常用更粗的分辨率
    coors_range_xyz=[0, -50, -3, 100, 50, 5],
    max_num_points_per_voxel=5,           # ← 练习 1 的 counts.max() 告诉你该设多少
    max_num_voxels=120_000,               # ← 由占用率算出来（第 3 节）
)(torch.from_numpy(pts))
#   ↑ 这两个上限设小了会**静默丢点**，而它们的正确值就是本 notebook 算的那两个数

# ── 3) 分辨率选择：把「精度要求」翻译成 r（练习 3）──
#   量化误差上界 = r√3/2，所以 r <= 2 * accuracy / √3
r_max_by_accuracy = 2 * required_accuracy_m / np.sqrt(3)
#   而内存约束给出下界：r >= (scene_volume / byte_budget) ** (1/3)
#   两者交集为空 -> 密集体素方案不可行，必须上稀疏（模块 03）或点云（模块 02）

# ── 4) 评测：3D 的 IoU 阈值不能照搬 2D（第 5 节）──
#   KITTI 用 0.7（车）/ 0.5（行人、骑车人）—— 而这个区分正是因为三个乘子：
#   同样的绝对误差，小目标的 IoU 掉得快得多（模块 05 会精确算出来）
```

> **落地顺序建议**：先用练习 1 算出你数据上的 `pts_per_voxel` 与 `counts.max()`
> （它们直接决定 `max_num_points_per_voxel` 与 `max_num_voxels` 该设多少，
> 而设错会静默丢点），再用练习 3 检查你的分辨率与精度要求是否自相矛盾。